# 02 – Integração entre fontes (JOINs)

Carrega os Parquet da pasta `bronze/`, integra os dados históricos e de Paris 2024, e gera os seguintes datasets na camada bronze:
- `medalhas_unificado` – quadro agregado por país e edição (histórico + Paris)
- `modalidades_paris2024` – contagem de medalhas por disciplina (Paris)
- `atletas_por_sexo_paris2024` – número de atletas únicos por gênero (Paris)

> Além disso, gera os arquivos JSON de metadados para cada um.

In [1]:
import pandas as pd
import json
from pathlib import Path
from datetime import date

BRONZE_DIR = Path("../bronze")
BRONZE_DIR.mkdir(exist_ok=True)


In [2]:
# Carregar os Parquet

df_hist = pd.read_parquet(BRONZE_DIR / "olympics_historico.parquet")
df_paris = pd.read_parquet(BRONZE_DIR / "olympics_paris2024.parquet")

print("Histórico – colunas:", list(df_hist.columns))
print("Paris – colunas:", list(df_paris.columns))


Histórico – colunas: ['year', 'edition', 'edition_id', 'country', 'country_noc', 'gold', 'silver', 'bronze', 'total']
Paris – colunas: ['medal_date', 'medal_type', 'medal_code', 'name', 'gender', 'country_code', 'country', 'country_long', 'nationality_code', 'nationality', 'nationality_long', 'team', 'team_gender', 'discipline', 'event', 'event_type', 'url_event', 'birth_date', 'code_athlete', 'code_team', 'is_medallist']


In [3]:
# 1. Tratamento do histórico
# O histórico já está agregado por país e edição. Extrair a temporada a partir do nome da edição.

df_hist['season'] = df_hist['edition'].apply(lambda x: x.split()[1] if ' ' in x else 'Summer')
df_hist_unified = df_hist[['year', 'edition', 'country', 'gold', 'silver', 'bronze', 'total', 'season']].copy()
print("Histórico preparado.")

Histórico preparado.


In [4]:
# 2. Agregar Paris por país e tipo de medalha
# Contar quantas medalhas de cada tipo por país

paris_agg = df_paris.groupby(['country', 'medal_type']).size().unstack(fill_value=0).reset_index()

# Renomear colunas para gold, silver, bronze
rename_map = {
    'Gold Medal': 'gold',
    'Silver Medal': 'silver',
    'Bronze Medal': 'bronze'
}
paris_agg = paris_agg.rename(columns=rename_map)

# Garantir que todas as colunas existam
for medal in ['gold', 'silver', 'bronze']:
    if medal not in paris_agg.columns:
        paris_agg[medal] = 0

paris_agg['total'] = paris_agg['gold'] + paris_agg['silver'] + paris_agg['bronze']
paris_agg['year'] = 2024
paris_agg['edition'] = '2024 Summer Olympics'
paris_agg['season'] = 'Summer'

paris_unified = paris_agg[['year', 'edition', 'country', 'gold', 'silver', 'bronze', 'total', 'season']].copy()
print(f"Paris agregado: {len(paris_unified)} países.")


Paris agregado: 92 países.


In [5]:
# 3. Unir histórico e Paris

df_all = pd.concat([df_hist_unified, paris_unified], ignore_index=True)
df_all = df_all.sort_values(['year', 'country']).reset_index(drop=True)
print(f"Dataset unificado: {len(df_all)} linhas.")

# Salvar
df_all.to_csv(BRONZE_DIR / "medalhas_unificado.csv", index=False)
df_all.to_parquet(BRONZE_DIR / "medalhas_unificado.parquet", index=False)
print("Salvo medalhas_unificado.parquet e .csv")

Dataset unificado: 1899 linhas.
Salvo medalhas_unificado.parquet e .csv


In [6]:
# 4. Dataset de modalidades (apenas Paris)

modalidades = df_paris.groupby(['discipline', 'medal_type']).size().reset_index(name='count')
modalidades.to_csv(BRONZE_DIR / "modalidades_paris2024.csv", index=False)
print("Salvo modalidades_paris2024.csv")


Salvo modalidades_paris2024.csv


In [7]:
# 5. Dataset de atletas por sexo (apenas Paris)

atletas_sexo = df_paris.groupby('gender')['name'].nunique().reset_index(name='num_atletas')
atletas_sexo.to_csv(BRONZE_DIR / "atletas_por_sexo_paris2024.csv", index=False)
print("Salvo atletas_por_sexo_paris2024.csv")

Salvo atletas_por_sexo_paris2024.csv


In [8]:
# 6. Gerar metadados para os novos datasets

def salvar_metadados(nome_dataset, fonte, descricao, campos_principais, observacoes, filename):
    metadados = {
        "nome_dataset": nome_dataset,
        "fonte": fonte,
        "descricao": descricao,
        "campos_principais": campos_principais,
        "data_criacao": date.today().isoformat(),
        "observacoes": observacoes
    }
    with open(BRONZE_DIR / filename, 'w', encoding='utf-8') as f:
        json.dump(metadados, f, indent=2, ensure_ascii=False)
    print(f"Metadados salvos: {filename}")

# Medalhas unificado
salvar_metadados(
    nome_dataset="Medalhas unificadas (histórico + Paris 2024)",
    fonte="Integração entre Base dos Dados (histórico) e Kaggle (medallists.csv de Paris)",
    descricao="Quadro de medalhas por país e edição, incluindo edições históricas (1896-2022) e Paris 2024. Formato agregado (ouro, prata, bronze, total).",
    campos_principais=["year", "edition", "country", "gold", "silver", "bronze", "total", "season"],
    observacoes="Os dados de Paris foram agregados a partir do medallists.csv. As colunas gold/silver/bronze representam a contagem de medalhas.",
    filename="medalhas_unificado.json"
)

# Modalidades Paris
salvar_metadados(
    nome_dataset="Medalhas por modalidade (Paris 2024)",
    fonte="Derivado do medallists.csv de Paris",
    descricao="Contagem de medalhas por disciplina e tipo de medalha (ouro, prata, bronze) nos Jogos de Paris 2024.",
    campos_principais=["discipline", "medal_type", "count"],
    observacoes="Cada linha representa quantas medalhas de um tipo foram concedidas em uma disciplina.",
    filename="modalidades_paris2024.json"
)

# Atletas por sexo Paris
salvar_metadados(
    nome_dataset="Atletas por gênero (Paris 2024)",
    fonte="Derivado do medallists.csv de Paris",
    descricao="Número de atletas únicos participantes, separados por gênero (Male/Female) nos Jogos de Paris 2024.",
    campos_principais=["gender", "num_atletas"],
    observacoes="Considera apenas atletas individuais registrados em medallists.csv. Equipes não são contabilizadas como atletas individuais.",
    filename="atletas_por_sexo_paris2024.json"
)

print("\n✅ Todos os metadados foram gerados.")

Metadados salvos: medalhas_unificado.json
Metadados salvos: modalidades_paris2024.json
Metadados salvos: atletas_por_sexo_paris2024.json

✅ Todos os metadados foram gerados.
